In [10]:
import heapq
import copy

class PuzzleState:
    def __init__(self, board, parent=None, move="", depth=0, cost=0):
        self.board = board
        self.parent = parent
        self.move = move
        self.depth = depth
        self.cost = cost

    def __lt__(self, other):
        return self.cost < other.cost

    def get_blank_pos(self):
        for r in range(3):
            for c in range(3):
                if self.board[r][c] == 0:
                    return r, c

    def get_neighbors(self):
        neighbors = []
        r, c = self.get_blank_pos()
        moves = {
            "Up": (r - 1, c),
            "Down": (r + 1, c),
            "Left": (r, c - 1),
            "Right": (r, c + 1)
        }

        for move_name, (nr, nc) in moves.items():
            if 0 <= nr < 3 and 0 <= nc < 3:
                new_board = copy.deepcopy(self.board)

                new_board[r][c], new_board[nr][nc] = new_board[nr][nc], new_board[r][c]
                neighbors.append(PuzzleState(new_board, self, move_name, self.depth + 1))
        return neighbors

    def to_tuple(self):
        return tuple(tuple(row) for row in self.board)

def manhattan_distance(board, goal_board):

    goal_pos = {}
    for r in range(3):
        for c in range(3):
            val = goal_board[r][c]
            goal_pos[val] = (r, c)

    distance = 0
    for r in range(3):
        for c in range(3):
            val = board[r][c]
            if val != 0:
                target_r, target_c = goal_pos[val]
                distance += abs(r - target_r) + abs(c - target_c)
    return distance

def solve_a_star(start_board, goal_board):
    start_state = PuzzleState(start_board)
    start_state.cost = manhattan_distance(start_board, goal_board)

    open_set = [start_state]
    closed_set = set()

    nodes_expanded = 0

    while open_set:
        current = heapq.heappop(open_set)
        nodes_expanded += 1

        if current.board == goal_board:

            path = []
            temp = current
            while temp:
                path.append(temp)
                temp = temp.parent
            return path[::-1], nodes_expanded

        closed_set.add(current.to_tuple())

        for neighbor in current.get_neighbors():
            if neighbor.to_tuple() in closed_set:
                continue

            neighbor.cost = neighbor.depth + manhattan_distance(neighbor.board, goal_board)
            heapq.heappush(open_set, neighbor)

    return None, nodes_expanded

def print_board(board):
    for row in board:
        print(" ".join(str(x) if x != 0 else "_" for x in row))
    print()

start_board = [
    [1, 2, 3],
    [5, 0, 6],
    [4, 7, 8]
]

goal_board = [
    [1, 2, 3],
    [4, 5, 6],
    [7, 8, 0]
]

print("Start State:")
print_board(start_board)
print("Goal State:")
print_board(goal_board)

path, nodes = solve_a_star(start_board, goal_board)

if path:
    print(f"Solvable! Path found in {len(path) - 1} steps. Nodes expanded: {nodes}\n")
    for i, state in enumerate(path):

        g_n = state.depth
        h_n = manhattan_distance(state.board, goal_board)
        f_n = g_n + h_n

        if i > 0:
            print(f"--- Move {i}: {state.move} ---")
        else:
            print("--- Initial State ---")

        print(f"g(n) = {g_n} (steps from start)")
        print(f"h(n) = {h_n} (Manhattan distance to goal)")
        print(f"f(n) = g(n) + h(n) = {f_n}\n")
        print_board(state.board)
else:
    print("No solution exists.")

Start State:
1 2 3
5 _ 6
4 7 8

Goal State:
1 2 3
4 5 6
7 8 _

Solvable! Path found in 4 steps. Nodes expanded: 5

--- Initial State ---
g(n) = 0 (steps from start)
h(n) = 4 (Manhattan distance to goal)
f(n) = g(n) + h(n) = 4

1 2 3
5 _ 6
4 7 8

--- Move 1: Left ---
g(n) = 1 (steps from start)
h(n) = 3 (Manhattan distance to goal)
f(n) = g(n) + h(n) = 4

1 2 3
_ 5 6
4 7 8

--- Move 2: Down ---
g(n) = 2 (steps from start)
h(n) = 2 (Manhattan distance to goal)
f(n) = g(n) + h(n) = 4

1 2 3
4 5 6
_ 7 8

--- Move 3: Right ---
g(n) = 3 (steps from start)
h(n) = 1 (Manhattan distance to goal)
f(n) = g(n) + h(n) = 4

1 2 3
4 5 6
7 _ 8

--- Move 4: Right ---
g(n) = 4 (steps from start)
h(n) = 0 (Manhattan distance to goal)
f(n) = g(n) + h(n) = 4

1 2 3
4 5 6
7 8 _

